# Wizard RL — ExperimentsLaunch the ablation and rebuild the figures from its results.The five configurations are additive: each adds exactly one axis to the onebefore it, so a difference in the outcome is attributable to that axis.| | bid_mode | value baseline | round sampling | tricks aux ||---|---|---|---|---|| **A** | policy | off | uniform | off || **B** | policy | **on** | uniform | off || **C** | policy | on | **r²** | off || **D** | policy | on | r² | **on** || **E** | **ev** | on | r² | on || **F** | ev | on | r² | on | *(plus heuristic training opponents)* |`p_heur = 0` for A–E: the story predates heuristic opponents, and mixing themin would add a variable that is not part of it. F is that variable on its own.**Runs are launched as subprocesses, not in notebook cells.** A run takeshours; a dying kernel must not take it with it.**Every configuration runs under three seeds.** A single run cannot tell a realeffect from a lucky initialisation; the figures show the mean with a min/maxband, the table shows mean +- half the range.

## Setup

In [ ]:
import json, os, subprocess, sys, timefrom pathlib import Pathimport numpy as npimport matplotlib.pyplot as pltREPO = Path.cwd()PYTHON = str(REPO / ".wizard_venv" / "bin" / "python")RESULTS = REPO / "results"LOGS = REPO / "run_logs"; LOGS.mkdir(exist_ok=True)sys.path.insert(0, str(REPO))from train import EXPERIMENTS, Config, load_resultsfor name, cfg in EXPERIMENTS.items():    if name != "default":        print(f"{name}: {cfg.run_suffix}")

## 1 — LaunchEach cell call starts one detached run. Stdout goes to `run_logs/<name>.log`,results to `results/<name>.json`, TensorBoard to the usual place.Start them one at a time if the machine is busy, or all at once if it is not —they are independent.

In [ ]:
SEEDS = (0, 1, 2)          # one seed is not evidence; three shows the spreaddef run_id(name, seed):    """Name of the result file for one (configuration, seed) pair."""    return f"{name}_seed{seed}"def launch(name, updates=None, seed=None):    """Start one configuration detached. Returns the Popen handle.    The command is built as a list of words -- exactly what you would type in    a terminal, only split up, so no shell and no quoting is involved:        launch("A")             ->  python train.py --config A        launch("A", seed=1)     ->  python train.py --config A --seed 1    Popen starts the process and returns immediately; the handle lets you ask    p.poll() (None while running), p.wait() or p.terminate(). With    start_new_session the run survives a kernel restart -- but the handle does    not, so use status() or running() to find them again.    """    cmd = [PYTHON, "train.py", "--config", name]    if updates is not None: cmd += ["--updates", str(updates)]    if seed is not None:    cmd += ["--seed", str(seed)]    tag = name if seed is None else run_id(name, seed)    log = open(LOGS / f"{tag}.log", "w")    p = subprocess.Popen(cmd, cwd=REPO, stdout=log, stderr=subprocess.STDOUT,                         start_new_session=True)    print(f"{tag} started, pid {p.pid}, log -> run_logs/{tag}.log")    return pdef launch_seeds(names="ABCDE", seeds=SEEDS, updates=None):    """Start every (configuration, seed) pair. Always seeds explicitly, so the    result files are named uniformly: results/A_seed0.json, A_seed1.json, ..."""    return {(n, s): launch(n, updates=updates, seed=s)            for n in names for s in seeds}def running():    """PIDs of training runs alive right now -- works without any handle, so    it still answers after a kernel restart."""    out = subprocess.run(["pgrep", "-af", "train.py --config"],                         capture_output=True, text=True).stdout.strip()    return out.splitlines() if out else []

In [ ]:
# Smoke first: two updates per pair, confirms every configuration runs at all.# procs = launch_seeds(updates=2)# The real thing -- 5 configurations x 3 seeds = 15 runs. Check the machine can# take that in parallel; otherwise start one configuration at a time.# procs = launch_seeds()

## 2 — Status

In [ ]:
def status(names="ABCDEF", seeds=SEEDS):    print(f"{'run':>10} {'evals':>6} {'update':>7} {'vs random':>10}")    for n in names:        for s in seeds:            res = load_results(run_id(n, s), result_dir=str(RESULTS))            if res is None or not res["history"]:                print(f"{run_id(n, s):>10} {'-':>6} {'-':>7} {'-':>10}"); continue            h = res["history"]; last = h[-1]            print(f"{run_id(n, s):>10} {len(h):>6} {last['update']:>7} "                  f"{last['score_vs_random']:>10.1f}")    alive = running()    print(f"\n{len(alive)} process(es) alive")    for line in alive:        print("  " + line)status()

## 3 — FiguresOne figure per row of the results table, each carrying exactly one claim.Everything is rebuilt from `results/*.json`, so the figures are reproducibleand no screenshot is involved.

In [ ]:
def series(name, key, seed):    """One run's curve for one metric."""    res = load_results(run_id(name, seed), result_dir=str(RESULTS))    if not res or not res["history"]:        return np.array([]), np.array([])    h = [r for r in res["history"] if key in r]    return (np.array([r["update"] for r in h]),            np.array([r[key] for r in h], dtype=float))def series_mean(name, key, seeds=SEEDS):    """Mean over seeds plus the min/max envelope.    Runs can be at different lengths (one still going, one stopped early), so    the curves are truncated to the shortest one before averaging -- averaging    over a different number of seeds at different x would bend the curve for    reasons that have nothing to do with training.    """    curves = [series(name, key, s) for s in seeds]    curves = [(x, y) for x, y in curves if len(x)]    if not curves:        return np.array([]), np.array([]), np.array([]), np.array([]), 0    n = min(len(x) for x, _ in curves)    xs = curves[0][0][:n]    Y = np.vstack([y[:n] for _, y in curves])    return xs, Y.mean(0), Y.min(0), Y.max(0), len(curves)def compare(key, title, ylabel, configs="ABCDE", seeds=SEEDS,            hline=None, hlabel=None, ax=None, band=True):    ax = ax or plt.subplots(figsize=(7, 4))[1]    for name in configs:        xs, mean, lo, hi, k = series_mean(name, key, seeds)        if not len(xs):            continue        line, = ax.plot(xs, mean, label=f"{name} (n={k})", linewidth=1.6)        if band and k > 1:            ax.fill_between(xs, lo, hi, alpha=.15, color=line.get_color())    if hline is not None:        ax.axhline(hline, ls="--", c="0.4", lw=1, label=hlabel)    ax.set_title(title); ax.set_xlabel("update"); ax.set_ylabel(ylabel)    ax.legend(); ax.grid(alpha=.3)    return ax

In [ ]:
# Headline: the axis that is never trained against.compare("score_vs_random", "Score against random opponents", "points / game",        hline=223.3, hlabel="heuristic reference (223.3)")plt.tight_layout(); plt.show()

In [ ]:
# A / C / E: the bidding ceiling in round 20. This is the central pathology,# and it now comes from the bids actually played, so it is comparable across# the policy configurations and the EV one.fig, axes = plt.subplots(1, 2, figsize=(13, 4))compare("bids_max_r20", "Highest bid in round 20", "bid", ax=axes[0])compare("bids_mean_r20", "Mean bid in round 20", "bid", ax=axes[1],        hline=6.67, hlabel="structural expectation 6.67")plt.tight_layout(); plt.show()

In [ ]:
# B and C: does the collapse move? The rank ratio is the share of the bid# logits' variation living in a single direction -- it rises as the head stops# distinguishing hands. Only defined where the bid head is a policy.fig, axes = plt.subplots(1, 2, figsize=(13, 4))compare("rank_bid_all", "Rank ratio of the bid logits (collapse)", "s0 / sum(s)",        configs="ABCD", ax=axes[0])compare("bids_std_r20", "Spread of the round-20 bids", "std", ax=axes[1],        hline=1.50, hlabel="spread of E[W|hand] (1.50)")plt.tight_layout(); plt.show()

In [ ]:
# D: how good is the tricks head at bid time? The bid in configuration E# depends entirely on it.compare("tricks_mae_r20_at_bid", "Trick prediction error at bid time, round 20",        "MAE (tricks)", configs="DE")plt.tight_layout(); plt.show()

In [ ]:
# E: the bid distribution over training. At update 0 the tricks head is# uninformative, q is uniform, and the EV rule has a fixed point near 10 --# that spike is a property of the scoring function, not a learned bid.res = load_results("E", result_dir=str(RESULTS))if res and res["bids_r20"]:    steps = sorted(res["bids_r20"], key=int)    show = [steps[0]] + steps[len(steps)//2::max(1, len(steps)//3)]    fig, ax = plt.subplots(figsize=(8, 4))    for s in dict.fromkeys(show):        ax.hist(res["bids_r20"][s], bins=np.arange(-0.5, 21.5), histtype="step",                density=True, linewidth=1.6, label=f"update {s}")    ax.axvline(6.67, ls="--", c="0.4", lw=1, label="expectation 6.67")    ax.set_title("Round-20 bid distribution over training")    ax.set_xlabel("bid"); ax.set_ylabel("density"); ax.legend(); ax.grid(alpha=.3)    plt.tight_layout(); plt.show()else:    print("no histograms yet -- they are written at every 250th update")

## 4 — Table for the READMEThe numbers behind the Experiments section, taken from the last eval point ofeach run.

In [ ]:
KEYS = [("score_vs_random", "vs random"), ("score_vs_heuristic", "vs heuristic"),        ("bids_max_r20", "max bid r20"), ("bids_mean_r20", "mean bid r20"),        ("bids_std_r20", "std bid r20"), ("acc_r20", "hit rate r20"),        ("tricks_mae_r20_at_bid", "tricks MAE r20")]def final_values(name, key, seeds=SEEDS):    """Last value of one metric, one number per seed."""    out = []    for s in seeds:        res = load_results(run_id(name, s), result_dir=str(RESULTS))        if res and res["history"]:            vals = [r[key] for r in res["history"] if key in r]            if vals:                out.append(vals[-1])    return outhdr = f"{'cfg':<4}" + "".join(f"{lbl:>20}" for _, lbl in KEYS)print(hdr); print("-" * len(hdr))for name in "ABCDEF":    vals = {k: final_values(name, k) for k, _ in KEYS}    if not any(vals.values()):        continue    row = f"{name:<4}"    for key, _ in KEYS:        v = vals[key]        # mean +- half the range, so a wide spread between seeds is visible        row += (f"{np.mean(v):>13.2f}+-{(max(v)-min(v))/2:<5.2f}" if len(v) > 1                else f"{v[0]:>20.2f}" if v else f"{'-':>20}")    print(row)